# 05. Projector LoRA and decoder + projector LoRA

The `mlp1` projector maps vision features into the language hidden space. Apply LoRA only to
its two Linear layers (indices 1 and 3), keeping LayerNorm and all original weights frozen.
This is projector **LoRA**, not full projector fine-tuning.

Compare projector-only against the decoder-only anchor, then decoder + projector. The combined
run starts from the original base rather than continuing the decoder checkpoint, making the
comparison about trainable parts. Match the winning decoder settings from 04 if you changed them.


In [ ]:
from pathlib import Path
import sys
import json
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "models.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "models.json").exists():
    raise FileNotFoundError("Start the kernel in 1pageTest or its 6_lora_training folder.")
sys.path.insert(0, str(ROOT / "6_lora_training"))
import lora_helpers as H
print("Interpreter:", sys.executable)
print("Dataset:", H.PAGE_DIR)
print("Local base model:", H.MODEL_DIR)


In [ ]:
pages = H.read_pages()
train_pages, val_pages = H.partition(pages, "train"), H.partition(pages, "val")
EXPERIMENTS = [
    H.experiment("projector_only_r4", decoder=None, projector=True),
    H.experiment("decoder_qv_last8_projector_r4", decoder="qv", last_layers=8, projector=True),
]
for config in EXPERIMENTS:
    directory = H.train_run(config, train_pages, val_pages)
    display(json.loads((directory / "summary.json").read_text()))


Expected first-step gradients: mlp1 for projector-only; mlp1 and language_model for the combined
run. Vision remains frozen. Compare generated ID accuracy, not just loss. Training the projector
without decoder LoRA can still improve visual grounding because gradients flow through the
frozen decoder into the trainable projector adapters.
